# Medical LLM Notebook 6 — Paired Statistical Validation

This notebook analyzes the 100 matched factual–hallucinated MedHallu pairs produced by Notebook 5.

**Primary analyses**
- Paired descriptive statistics
- Wilcoxon signed-rank tests
- Holm correction for multiple testing
- Rank-biserial effect size
- Paired Cohen's \(d_z\)
- Bootstrap 95% confidence intervals
- Response-length sensitivity
- Length-adjusted difference regression with HC3 robust standard errors
- Length-matched sensitivity analysis
- Feature redundancy analysis

`Words` is treated as a **control variable**, not as a hallucination feature.


In [ ]:
!pip -q install scipy statsmodels

import os
import zipfile
import numpy as np
import pandas as pd
import scipy.stats as st
import statsmodels.api as sm

from scipy.stats import rankdata, wilcoxon, spearmanr, shapiro
from statsmodels.stats.multitest import multipletests
from google.colab import files

SEED = 42
rng = np.random.default_rng(SEED)

print("Notebook 6 libraries loaded.")

## 1. Upload Notebook 5 outputs

In [ ]:
uploaded = files.upload()

zip_files = [
    name for name in uploaded.keys()
    if name.lower().endswith(".zip")
]

assert len(zip_files) >= 1, "Upload Notebook5_100Pair_Outputs.zip"

ZIP_FILE = zip_files[0]
EXTRACT_DIR = "/content/Notebook6_input"

os.makedirs(EXTRACT_DIR, exist_ok=True)

with zipfile.ZipFile(ZIP_FILE, "r") as z:
    z.extractall(EXTRACT_DIR)

print("Extracted:", ZIP_FILE)

In [ ]:
import glob

paired_file = glob.glob(
    "/content/Notebook6_input/**/MedHallu_100_Paired_Differences.csv",
    recursive=True
)[0]

response_file = glob.glob(
    "/content/Notebook6_input/**/MedHallu_200_Response_Features.csv",
    recursive=True
)[0]

paired = pd.read_csv(paired_file)
responses = pd.read_csv(response_file)

print("Paired data:", paired.shape)
print("Response data:", responses.shape)

assert len(paired) == 100
assert paired["Pair_ID"].nunique() == 100
assert len(responses) == 200
assert responses["Pair_ID"].nunique() == 100

print("Validation passed.")

## 2. Define analysis features

In [ ]:
FEATURES = [
    "GCC",
    "Communication_Efficiency",
    "Mean_Edge_Weight",
    "Max_PageRank",
    "PageRank_Concentration",
    "Mean_Betweenness",
    "Max_Betweenness",
    "Max_GCCDC",
    "Mean_Structural_Influence",
    "Max_Structural_Influence",
    "QAS_Mean",
    "QAS_Median",
    "QAS_Std"
]

CONTROL = "Words"

print("Primary features:", len(FEATURES))
print("Control variable:", CONTROL)

## 3. Paired statistical tests and effect sizes

In [ ]:
def signed_rank_biserial(delta):
    delta = np.asarray(delta, dtype=float)
    delta = delta[np.isfinite(delta)]
    delta = delta[delta != 0]

    if len(delta) == 0:
        return np.nan

    ranks = rankdata(np.abs(delta))
    w_pos = ranks[delta > 0].sum()
    w_neg = ranks[delta < 0].sum()

    return (w_pos - w_neg) / (w_pos + w_neg)


def bootstrap_ci(values, statistic=np.mean, B=10000, seed=42):
    values = np.asarray(values, dtype=float)
    values = values[np.isfinite(values)]

    local_rng = np.random.default_rng(seed)
    n = len(values)

    boot = np.empty(B)

    for b in range(B):
        sample = local_rng.choice(values, size=n, replace=True)
        boot[b] = statistic(sample)

    return np.quantile(boot, [0.025, 0.975])


rows = []

for feature in FEATURES:

    factual = paired[f"{feature}_Factual"].astype(float).to_numpy()
    hallucinated = paired[f"{feature}_Hallucinated"].astype(float).to_numpy()
    delta = hallucinated - factual

    w = wilcoxon(
        hallucinated,
        factual,
        alternative="two-sided",
        zero_method="wilcox",
        method="auto"
    )

    sd_delta = np.std(delta, ddof=1)
    cohen_dz = (
        np.mean(delta) / sd_delta
        if sd_delta > 0 else np.nan
    )

    mean_ci = bootstrap_ci(delta, np.mean, seed=SEED)
    median_ci = bootstrap_ci(delta, np.median, seed=SEED)

    rows.append({
        "Feature": feature,
        "Factual_Mean": np.mean(factual),
        "Hallucinated_Mean": np.mean(hallucinated),
        "Mean_Delta_HminusF": np.mean(delta),
        "Mean_Delta_CI_Low": mean_ci[0],
        "Mean_Delta_CI_High": mean_ci[1],
        "Median_Delta_HminusF": np.median(delta),
        "Median_Delta_CI_Low": median_ci[0],
        "Median_Delta_CI_High": median_ci[1],
        "Wilcoxon_W": w.statistic,
        "Wilcoxon_p": w.pvalue,
        "Rank_Biserial": signed_rank_biserial(delta),
        "Cohen_dz": cohen_dz,
        "Shapiro_Delta_p": shapiro(delta).pvalue,
        "Positive_Delta_n": int((delta > 0).sum()),
        "Negative_Delta_n": int((delta < 0).sum()),
        "Zero_Delta_n": int((delta == 0).sum())
    })

primary_stats = pd.DataFrame(rows)

primary_stats["Wilcoxon_p_Holm"] = multipletests(
    primary_stats["Wilcoxon_p"],
    method="holm"
)[1]

primary_stats["Significant_Holm_0.05"] = (
    primary_stats["Wilcoxon_p_Holm"] < 0.05
)

display(
    primary_stats
    .sort_values("Wilcoxon_p_Holm")
    .round(6)
)

## 4. Response-length sensitivity

This is essential because several graph statistics can depend on the number of response words. We test:

1. Across all 200 responses: Spearman correlation between each feature and `Words`.
2. Across 100 matched pairs: Spearman correlation between \(\Delta\)Feature and \(\Delta\)Words.


In [ ]:
size_rows = []

for feature in FEATURES:

    rho_all, p_all = spearmanr(
        responses["Words"],
        responses[feature]
    )

    rho_delta, p_delta = spearmanr(
        paired["Delta_Words"],
        paired[f"Delta_{feature}"]
    )

    size_rows.append({
        "Feature": feature,
        "rho_Feature_vs_Words_200responses": rho_all,
        "p_Feature_vs_Words": p_all,
        "rho_DeltaFeature_vs_DeltaWords": rho_delta,
        "p_DeltaFeature_vs_DeltaWords": p_delta
    })

length_sensitivity = pd.DataFrame(size_rows)

length_sensitivity["p_Feature_vs_Words_Holm"] = multipletests(
    length_sensitivity["p_Feature_vs_Words"],
    method="holm"
)[1]

length_sensitivity["p_Delta_vs_DeltaWords_Holm"] = multipletests(
    length_sensitivity["p_DeltaFeature_vs_DeltaWords"],
    method="holm"
)[1]

display(
    length_sensitivity
    .sort_values(
        "rho_Feature_vs_Words_200responses",
        key=lambda s: s.abs(),
        ascending=False
    )
    .round(6)
)

## 5. Length-adjusted paired analysis

A pair fixed-effect model reduces to the difference regression

\[
\Delta X_i = \beta_0 + \beta_1 \Delta Words_i + \varepsilon_i.
\]

Here, \(eta_0\) estimates the factual–hallucinated feature difference when the paired word-count difference is zero. HC3 robust standard errors are used.


In [ ]:
adjusted_rows = []

delta_words = paired["Delta_Words"].astype(float).to_numpy()

for feature in FEATURES:

    delta_feature = paired[f"Delta_{feature}"].astype(float).to_numpy()

    X = sm.add_constant(delta_words)

    model = sm.OLS(
        delta_feature,
        X
    ).fit(
        cov_type="HC3"
    )

    adjusted_rows.append({
        "Feature": feature,
        "Adjusted_Intercept": model.params[0],
        "Adjusted_Intercept_SE": model.bse[0],
        "Adjusted_Intercept_p": model.pvalues[0],
        "Length_Beta": model.params[1],
        "Length_Beta_SE": model.bse[1],
        "Length_Beta_p": model.pvalues[1],
        "R_squared": model.rsquared
    })

length_adjusted = pd.DataFrame(adjusted_rows)

length_adjusted["Adjusted_Intercept_p_Holm"] = multipletests(
    length_adjusted["Adjusted_Intercept_p"],
    method="holm"
)[1]

length_adjusted["Length_Beta_p_Holm"] = multipletests(
    length_adjusted["Length_Beta_p"],
    method="holm"
)[1]

length_adjusted["Adjusted_Significant_Holm_0.05"] = (
    length_adjusted["Adjusted_Intercept_p_Holm"] < 0.05
)

display(
    length_adjusted
    .sort_values("Adjusted_Intercept_p_Holm")
    .round(6)
)

## 6. Length-matched sensitivity analysis

In [ ]:
MATCH_THRESHOLD = 10

matched = paired[
    paired["Delta_Words"].abs() <= MATCH_THRESHOLD
].copy()

print(
    f"Pairs with |Delta Words| <= {MATCH_THRESHOLD}:",
    len(matched)
)

matched_rows = []

for feature in FEATURES:

    f = matched[f"{feature}_Factual"].astype(float)
    h = matched[f"{feature}_Hallucinated"].astype(float)
    d = h - f

    if len(matched) > 0:
        w = wilcoxon(h, f)
        p = w.pvalue
    else:
        p = np.nan

    matched_rows.append({
        "Feature": feature,
        "n_pairs": len(matched),
        "Mean_Delta": d.mean(),
        "Median_Delta": d.median(),
        "Wilcoxon_p": p
    })

matched_sensitivity = pd.DataFrame(matched_rows)

matched_sensitivity["Wilcoxon_p_Holm"] = multipletests(
    matched_sensitivity["Wilcoxon_p"],
    method="holm"
)[1]

display(
    matched_sensitivity
    .sort_values("Wilcoxon_p_Holm")
    .round(6)
)

## 7. Difficulty-level descriptive sensitivity

In [ ]:
print("Difficulty counts:")
display(paired["Difficulty"].value_counts())

difficulty_rows = []

for difficulty, group in paired.groupby("Difficulty"):

    for feature in FEATURES:

        d = group[f"Delta_{feature}"].astype(float)

        difficulty_rows.append({
            "Difficulty": difficulty,
            "Feature": feature,
            "n": len(group),
            "Mean_Delta": d.mean(),
            "Median_Delta": d.median(),
            "Positive_n": int((d > 0).sum()),
            "Negative_n": int((d < 0).sum())
        })

difficulty_summary = pd.DataFrame(difficulty_rows)

display(
    difficulty_summary.round(6)
)

## 8. Feature redundancy

In [ ]:
delta_columns = [
    f"Delta_{feature}"
    for feature in FEATURES
]

delta_correlation = paired[
    delta_columns
].corr(
    method="spearman"
)

display(
    delta_correlation.round(3)
)

## 9. Robust-candidate summary

In [ ]:
candidate_summary = (
    primary_stats[
        [
            "Feature",
            "Mean_Delta_HminusF",
            "Median_Delta_HminusF",
            "Wilcoxon_p_Holm",
            "Rank_Biserial",
            "Cohen_dz"
        ]
    ]
    .merge(
        length_adjusted[
            [
                "Feature",
                "Adjusted_Intercept",
                "Adjusted_Intercept_p_Holm"
            ]
        ],
        on="Feature"
    )
    .merge(
        length_sensitivity[
            [
                "Feature",
                "rho_Feature_vs_Words_200responses",
                "rho_DeltaFeature_vs_DeltaWords"
            ]
        ],
        on="Feature"
    )
)

candidate_summary[
    "Raw_Paired_Significant"
] = (
    candidate_summary[
        "Wilcoxon_p_Holm"
    ] < 0.05
)

candidate_summary[
    "Length_Adjusted_Significant"
] = (
    candidate_summary[
        "Adjusted_Intercept_p_Holm"
    ] < 0.05
)

display(
    candidate_summary
    .sort_values(
        [
            "Length_Adjusted_Significant",
            "Wilcoxon_p_Holm"
        ],
        ascending=[
            False,
            True
        ]
    )
    .round(6)
)

## 10. Save Notebook 6 outputs

In [ ]:
OUTPUT_DIR = "/content/Notebook6_outputs"
os.makedirs(OUTPUT_DIR, exist_ok=True)

primary_stats.to_csv(
    f"{OUTPUT_DIR}/Paired_Primary_Statistics.csv",
    index=False
)

length_sensitivity.to_csv(
    f"{OUTPUT_DIR}/Length_Sensitivity.csv",
    index=False
)

length_adjusted.to_csv(
    f"{OUTPUT_DIR}/Length_Adjusted_Regression.csv",
    index=False
)

matched_sensitivity.to_csv(
    f"{OUTPUT_DIR}/Length_Matched_Sensitivity.csv",
    index=False
)

difficulty_summary.to_csv(
    f"{OUTPUT_DIR}/Difficulty_Descriptive_Summary.csv",
    index=False
)

delta_correlation.to_csv(
    f"{OUTPUT_DIR}/Delta_Feature_Correlation.csv"
)

candidate_summary.to_csv(
    f"{OUTPUT_DIR}/Candidate_Feature_Summary.csv",
    index=False
)

print("Notebook 6 outputs saved.")

In [ ]:
!zip -r /content/Notebook6_Statistical_Outputs.zip /content/Notebook6_outputs

files.download(
    "/content/Notebook6_Statistical_Outputs.zip"
)